## 1. 라이브러리 로드

In [23]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

---
## 2. 데이터 로드

In [24]:
path = "../../../tft_game_dataset/TFT_Platinum_MatchData.csv"
path2 = "../../../tft_game_dataset/TFT_Diamond_MatchData.csv"
path3 = "../../../tft_game_dataset/TFT_Master_MatchData.csv"
path4 = "../../../tft_game_dataset/TFT_GrandMaster_MatchData.csv"
path5 = "../../../tft_game_dataset/TFT_Challenger_MatchData.csv"
path6 = "../../../tft_game_dataset/TFT_Champion_CurrentVersion.csv"
path7 = "../../../tft_game_dataset/TFT_Item_CurrentVersion.csv"

df_platinum_Match = pd.read_csv(path)
df_diamond_Match = pd.read_csv(path2)
df_master_Match = pd.read_csv(path3)
df_grand_master_Match = pd.read_csv(path4)
df_challenger_Match = pd.read_csv(path5)

df_champion_info = pd.read_csv(path6)
df_items_info = pd.read_csv(path7)


In [25]:
# 매치관련 데이터가 담긴 딕셔너리 정의
match_data = {
    'platinum': df_platinum_Match,
    'diamond': df_diamond_Match,
    'master': df_master_Match,
    'grand_master': df_grand_master_Match,
    'challenger': df_challenger_Match,
}

# 각 티어별 테이블에 'Tier'정보가 담긴 컬럼 추가
for name, df in match_data.items():
    df['tier'] = name


# 모든 티어의 경기데이터가 담긴 데이터프레임 제작
# ignore_index=True: 데이터를 병합한 뒤, 새로운 인덱스 부여
df_all_match = pd.concat(match_data.values(), ignore_index=True)

df_all_match.head(3)

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Mercenary': 1, 'Set3_Blademaster': 1, 'Set3_Mystic': 1, 'Valkyrie': 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum


---
## 3. 데이터 전처리
### 3-1. 중복행 제거

In [26]:
# 중복행 제거
duplicates = df_all_match[df_all_match.duplicated(keep=False)]

print(f"중복 제거 전: {len(df_all_match)}")
print(f"중복된 행 개수: {len(duplicates)}")

# 결과 확인
df_all_match = df_all_match.drop_duplicates()
print(f"\n중복 제거 후: {len(df_all_match)}")

중복 제거 전: 399998
중복된 행 개수: 80

중복 제거 후: 399930


### 3-2. 컬럼명 소문자로 통일

In [27]:
# 전체 컬럼명 소문자 통일
df_all_match.columns = df_all_match.columns.str.lower()

print(df_all_match.columns)

Index(['gameid', 'gameduration', 'level', 'lastround', 'ranked',
       'ingameduration', 'combination', 'champion', 'tier'],
      dtype='str')


### 3-3. ranked=0인 데이터 삭제

In [28]:
# ranked = 0이 포함된 경기 데이터 삭제
df_all_match_2 = df_all_match.copy()

# ranked==0인 행의 gameId 추출
drop_game_ids = df_all_match_2[df_all_match_2['ranked']==0]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx)

print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: {df_all_match.shape}")
print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: (399930, 9)
ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-4. 경기시간 = 0인 데이터 삭제

In [29]:
# 전체 게임시간 = 0인 행의 GameId 추출
zero_duration_ids = set(df_all_match_2[df_all_match_2['gameduration'] == 0]['gameid'])

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_2 = df_all_match_2[df_all_match_2['gameid'].isin(zero_duration_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_2)

print(f"gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-5. 경기 참여 유저 수가 8명 미만인 게임 삭제

In [30]:
# 게임 ID별 플레이어 수 정보 추가
df_all_match_2['player_cnt'] = df_all_match_2['gameid'].map(df_all_match_2['gameid'].value_counts())

# player_cnt < 8인 행의 gameId 추출
drop_game_ids_3 = df_all_match_2[df_all_match_2['player_cnt'] < 8]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_3 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_3)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_3)

print(f"player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: (399872, 10)


### 3-6. 시즌2 데이터 삭제

In [31]:
# 시즌 2 고유 시너지 목록
season2_keys = {'Alchemist', 'Avatar', 'Berserker',
                'Crystal', 'Desert', 'Druid', 'Electric',
                'Light', 'Mage', 'Mountain', 'Mystic', 'Ocean',
                'Poison', 'Predator', 'Set2_Assassin', 'Set2_Blademaster',
                'Set2_Glacial', 'Set2_Ranger', 'Shadow', 'Soulbound',
                'Summoner', 'Warden', 'Wind', 'Woodland'}  # 시즌 2 시너지 입력


# 시즌 2 시너지가 하나라도 포함되면 시즌 2로 분류
df_all_match_2['season'] = df_all_match_2['combination'].apply(
    lambda x: 'season 2' if any(k in season2_keys for k in ast.literal_eval(x).keys())
    else 'season 3'
)

# season 2인 행의 gameId 추출
drop_game_ids_4 = df_all_match_2[df_all_match_2['season'] == 'season 2']['gameid'].unique()


# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_4 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_4)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_4)

print(f"season 2인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

season 2인 데이터가 포함된 경기 데이터 삭제한 후: (396640, 11)


### 3-7. 시즌 3 시너지 더미 데이터 ”TemplateTrait” 삭제
- 시너지 데이터 중 해당 값인 `{'TemplateTrait': 1}`만 삭제

In [32]:
# TemplateTrait 키가 있는 행만 필터링 후 값 확인
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys() # TemplateTrait 키가 있는 행만 필터링
)]['combination'].apply(                                    # 필터링된 행 중에서 combination 컬럼만 가져옴
    lambda x: ast.literal_eval(x).get('TemplateTrait')      # TemplateTrait 키에 할당된 값을 가져옴
).value_counts()


combination
1    65987
Name: count, dtype: int64

In [33]:
# 분석 목적에 영향을 주지 않는 dummy 데이터로 판단하여 시너지 중 'TemplateTrait'만 삭제
df_all_match_2['combination'] = df_all_match_2['combination'].apply(
    lambda x: json.dumps(
        # key: value = 키-값의 쌍을 의미함
        {key: value for key, value in ast.literal_eval(x).items() if key != 'TemplateTrait'},
        ensure_ascii=False
    )
)

In [34]:
# TemplateTrait 키가 있는 행 수 확인 → (0,11)이면 완전히 삭제된 것
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys()
)].shape

(0, 11)

### 3-8. 유저 ID 컬럼 제작

In [35]:
df_all_match_2['user_id'] = 'KR-USER-' + (df_all_match_2.index + 1).astype(str)

In [36]:
df_all_match_2['user_id']

0              KR-USER-1
1              KR-USER-2
2              KR-USER-3
3              KR-USER-4
4              KR-USER-5
               ...      
399993    KR-USER-399994
399994    KR-USER-399995
399995    KR-USER-399996
399996    KR-USER-399997
399997    KR-USER-399998
Name: user_id, Length: 396640, dtype: str

### 3-9. 티어가 섞인 경기 정보 추출

In [37]:
# 티어가 섞인 gameId 추출
mixed_gameids = df_all_match_2.groupby('gameid')['tier'].nunique() # gameid별로 tier의 고유값 개수를 계산
mixed_gameids = mixed_gameids[mixed_gameids > 1].index # 고유값이 2 이상인 gameId만 추출

# 해당 gameId의 티어 구성 확인
# mixed_gameids에 포함된 gameId를 가진 행만 추출 -> gameid별로 어떤 티어가 몇 명씩 있는지 카운트
df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['tier'].value_counts()

gameid         tier        
KR_4263820118  platinum        8
               master          8
KR_4313697214  platinum        8
               master          8
KR_4320079059  platinum        8
               diamond         8
KR_4344513862  diamond         8
               master          8
KR_4347884427  diamond         8
               master          8
KR_4357966612  platinum        8
               grand_master    8
KR_4358922415  diamond         8
               master          8
KR_4361594426  diamond         8
               master          8
KR_4361773461  diamond         8
               grand_master    8
KR_4362846604  diamond         8
               master          8
KR_4364453473  diamond         8
               grand_master    8
KR_4365284161  diamond         8
               master          8
KR_4378896137  platinum        8
               diamond         8
KR_4381231118  platinum        8
               diamond         8
KR_4387025966  platinum        8
               

In [38]:
# 티어 순서 정의 (숫자가 높을수록 높은 티어)
tier_order = {
    'platinum': 1,
    'diamond': 2,
    'master': 3,
    'grand_master': 4,
    'challenger': 5
}

### 여기서부턴 제겁니다 껄껄

In [39]:
import json
import ast
from collections import Counter

# 1. 챔피언 정보 테이블의 시너지명 -> 실제 게임 데이터(combination)용 시너지명으로 변환하는 매퍼
trait_mapper = {
    'Space Pirate': 'SpacePirate',
    'Star Guardian': 'StarGuardian',
    'Celestial': 'Set3_Celestial',
    'Dark Star': 'DarkStar',
    'Mech-Pilot': 'MechPilot',
    'Cybernetic': 'Cybernetic',
    'Rebel': 'Rebel',
    'Valkyrie': 'Valkyrie',
    'Void': 'Set3_Void',
    'Chrono': 'Chrono',
    'Mercenary': 'Mercenary',
    'Demolitionist': 'Demolitionist',
    'Blaster': 'Blaster',
    'Protector': 'Protector',
    'Mana-Reaver': 'ManaReaver',
    'Sorcerer': 'Set3_Sorcerer',
    'Vanguard': 'Vanguard',
    'Mystic': 'Set3_Mystic',
    'Blademaster': 'Set3_Blademaster',
    'Brawler': 'Set3_Brawler',
    'Infiltrator': 'Infiltrator',
    'Infiltrato': 'Infiltrator',  # 원본 데이터 오타(에코 클래스) 교정
    'Starship': 'Starship',
    'Sniper': 'Sniper'
}

# 2. 아이템 번호별 부여되는 시너지 (뒤집개류 수동 매핑)
spatula_items = {
    18: 'Set3_Blademaster', # 몰락한 왕의 검
    28: 'Infiltrator',      # 잠입자의 발톱
    38: 'Demolitionist',    # 폭파광의 화약
    48: 'StarGuardian',     # 별 수호자의 부적
    58: 'Rebel',            # 반군 훈장
    68: 'Set3_Celestial',   # 천상의 구
    78: 'Protector',        # 수호자의 흉갑
    89: 'DarkStar'          # 암흑의 별 심장
}

# 3. 비어있는 combination 복구 함수 정의
def restore_combination(row, df_champ):
    # 3-1. 이미 시너지가 정상적으로 있다면 기존 값 그대로 유지
    if row['combination'] != '{}' and row['combination'] != '':
        return row['combination']
    
    # 3-2. 올린 챔피언마저 비어있으면 빈 딕셔너리 반환
    if row['champion'] == '{}' or row['champion'] == '':
        return '{}'
    
    try:
        champs_in_game = ast.literal_eval(row['champion'])
    except:
        return '{}'
        
    current_synergies = Counter()
    
    for champ_name, info in champs_in_game.items():
        # 챔피언 이름 대소문자 매칭 (경기 데이터는 'Fiora', 챔피언 테이블은 'fiora'로 되어 있음)
        champ_data = df_champ[df_champ['name'] == champ_name.lower()]
        
        if not champ_data.empty:
            origin = champ_data['origin'].iloc[0]
            classes = champ_data['class'].iloc[0]
            
            traits = []
            traits.append(origin) # 계열(Origin) 추가
            
            # 직업(Class) 추가: 문자열로 된 리스트 처리
            if isinstance(classes, str):
                try:
                    classes = ast.literal_eval(classes)
                except:
                    classes = [classes]
            elif not isinstance(classes, list):
                classes = [classes]
                
            traits.extend(classes)
            
            # 매퍼를 거쳐서 실제 combination에 쓰이는 포맷으로 변환 후 카운트 추가
            for t in traits:
                mapped_trait = trait_mapper.get(t, t)
                current_synergies[mapped_trait] += 1
        
        # 3-3. 아이템(뒤집개)에 의한 추가 시너지 합산
        for item_id in info.get('items', []):
            if item_id in spatula_items:
                granted_trait = spatula_items[item_id]
                current_synergies[granted_trait] += 1

    # 복구된 시너지가 하나도 없으면 '{}' 반환
    if not current_synergies:
        return '{}'
    
    # 기존 데이터 양식과 동일하게 JSON 문자열 형태로 반환
    return json.dumps(dict(current_synergies), ensure_ascii=False)

# 4. 함수 적용 (전체 데이터에 실행)
df_all_match_2['combination'] = df_all_match_2.apply(
    lambda row: restore_combination(row, df_champion_info), axis=1
)

# 5. 복구 결과 확인
restored_count = df_all_match_2[df_all_match_2['combination'] != '{}'].shape[0]
total_count = df_all_match_2.shape[0]
print(f"✅ 복구 완료! 전체 {total_count}건 중, 정상적인 시너지 데이터가 {restored_count}건 확보되었습니다.")

✅ 복구 완료! 전체 396640건 중, 정상적인 시너지 데이터가 396377건 확보되었습니다.


In [40]:
import ast
from collections import Counter
import pandas as pd

spatula_items = {
    18: 'Set3_Blademaster',
    28: 'Infiltrator',
    38: 'Demolitionist',
    48: 'StarGuardian',
    58: 'Rebel',
    68: 'Set3_Celestial',
    78: 'Protector',
    89: 'DarkStar'
}

spatula_item_ids = set(spatula_items.keys())
spatula_counter = Counter()
users_with_spatula = 0

for champ_str in df_all_match_2['champion']:
    if pd.isna(champ_str) or champ_str == '{}':
        continue
        
    try:
        champs = ast.literal_eval(champ_str)
        has_spatula_in_this_match = False
        
        for champ_info in champs.values():
            items = champ_info.get('items', [])
            for item in items:
                if item in spatula_item_ids:
                    spatula_counter[item] += 1
                    has_spatula_in_this_match = True
                    
        if has_spatula_in_this_match:
            users_with_spatula += 1
    except:
        continue

print(f"시너지 아이템 장착 유저 수: {users_with_spatula}")
for item_id, count in spatula_counter.most_common():
    print(f"아이템 ID {item_id} ({spatula_items[item_id]}): {count}회")

시너지 아이템 장착 유저 수: 139604
아이템 ID 38 (Demolitionist): 35313회
아이템 ID 28 (Infiltrator): 26021회
아이템 ID 68 (Set3_Celestial): 23978회
아이템 ID 48 (StarGuardian): 19115회
아이템 ID 58 (Rebel): 18144회
아이템 ID 78 (Protector): 18108회
아이템 ID 18 (Set3_Blademaster): 14633회
아이템 ID 89 (DarkStar): 14246회


In [41]:
import ast
import json
import pandas as pd

# 1. 시너지별 활성화 기준(Threshold) 정의
synergy_thresholds = {
    # 직업 시너지
    'Set3_Blademaster': [3, 6, 9],
    'ManaReaver': [2],
    'Set3_Sorcerer': [2, 4, 6, 8],
    'Vanguard': [2, 4],
    'Protector': [2, 4, 6],
    'Set3_Mystic': [2, 4],
    'Set3_Brawler': [2, 4],
    'Mercenary': [1],
    'Starship': [1],
    'Infiltrator': [2, 4, 6],
    'Sniper': [2],
    'Blaster': [2, 4],
    'Demolitionist': [2],
    
    # 계열 시너지
    'Set3_Void': [3],
    'MechPilot': [3],
    'Rebel': [3, 6, 9],
    'Valkyrie': [2],
    'StarGuardian': [3, 6],
    'Cybernetic': [3, 6],
    'Chrono': [2, 4, 6],
    'DarkStar': [3, 6, 9],
    'SpacePirate': [2, 4],
    'Set3_Celestial': [2, 4, 6]
}

# 2. 활성화 레벨 계산 함수
def get_active_level(count, thresholds):
    active = 0
    for t in thresholds:
        if count >= t:
            active = t
        else:
            break
    return active

# 3. 딕셔너리(JSON) 형태로 활성화된 시너지만 반환하는 함수
def parse_active_synergies_to_dict(comb_str):
    if pd.isna(comb_str) or comb_str == '{}':
        return '{}'
        
    try:
        comb_dict = ast.literal_eval(comb_str)
    except:
        return '{}'
        
    active_res = {}
    for syn, count in comb_dict.items():
        if syn in synergy_thresholds:
            active_lvl = get_active_level(count, synergy_thresholds[syn])
            # 활성화 레벨이 0보다 큰 경우(최소 조건 달성)에만 딕셔너리에 추가
            if active_lvl > 0:
                active_res[syn] = active_lvl
                
    # 활성화된 시너지가 하나도 없으면 빈 딕셔너리 문자열 반환
    if not active_res:
        return '{}'
        
    # 기존 combination 컬럼과 동일하게 JSON 문자열 형태로 반환
    return json.dumps(active_res, ensure_ascii=False)

# 4. 파생 변수 컬럼 'active_synergy' 생성
df_all_match_2['active_synergy'] = df_all_match_2['combination'].apply(parse_active_synergies_to_dict)

# ✅ 결과 확인 (기존 combination과 변환된 active_synergy 비교)
display(df_all_match_2[['combination', 'active_synergy']].head(10))

,combination,active_synergy
0,"{""Cybernetic"": 1, ""Demolitionist"": 1, ""Infiltrator"": 1, ""Rebel"": 1, ""Set3_Brawler"": 1, ""Set3_Celestial"": 1, ""Set3_Void"": 1, ""Sniper"": 1}",{}
1,"{""Blaster"": 1, ""Chrono"": 1, ""Cybernetic"": 4, ""Demolitionist"": 1, ""Rebel"": 1, ""Set3_Blademaster"": 2, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 1, ""Set3_Void"": 1, ""Valkyrie"": 1, ""Vanguard"": 2}","{""Cybernetic"": 3, ""Vanguard"": 2}"
2,"{""Blaster"": 1, ""Cybernetic"": 1, ""DarkStar"": 2, ""Infiltrator"": 1, ""Mercenary"": 1, ""Set3_Blademaster"": 1, ""Set3_Mystic"": 1, ""Valkyrie"": 1}","{""Mercenary"": 1}"
3,"{""DarkStar"": 1, ""Protector"": 2, ""Set3_Blademaster"": 1, ""Set3_Celestial"": 5, ""Set3_Mystic"": 1, ""Sniper"": 1, ""StarGuardian"": 2, ""Vanguard"": 2}","{""Protector"": 2, ""Set3_Celestial"": 4, ""Vanguard"": 2}"
4,"{""Blaster"": 1, ""Chrono"": 5, ""DarkStar"": 3, ""Protector"": 1, ""Set3_Blademaster"": 1, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 2, ""Sniper"": 2}","{""Chrono"": 4, ""DarkStar"": 3, ""Set3_Sorcerer"": 2, ""Sniper"": 2}"
5,"{""Rebel"": 1, ""Set3_Blademaster"": 1, ""Set3_Celestial"": 1, ""Sniper"": 1, ""StarGuardian"": 1, ""Vanguard"": 1}",{}
6,"{""Protector"": 1, ""Set3_Mystic"": 1, ""Set3_Sorcerer"": 3, ""StarGuardian"": 6, ""Vanguard"": 1}","{""Set3_Sorcerer"": 2, ""StarGuardian"": 6}"
7,"{""Blaster"": 1, ""Cybernetic"": 1, ""Infiltrator"": 1, ""Set3_Blademaster"": 1, ""Set3_Celestial"": 1, ""Set3_Sorcerer"": 1, ""StarGuardian"": 1, ""Valkyrie"": 1}",{}
8,"{""Blaster"": 1, ""DarkStar"": 2, ""Set3_Celestial"": 1, ""Set3_Sorcerer"": 1, ""Set3_Void"": 1, ""Sniper"": 2, ""SpacePirate"": 1, ""StarGuardian"": 1, ""Vanguard"": 2}","{""Sniper"": 2, ""Vanguard"": 2}"
9,"{""Chrono"": 4, ""DarkStar"": 3, ""Infiltrator"": 1, ""Rebel"": 1, ""Set3_Blademaster"": 1, ""Set3_Brawler"": 2, ""Set3_Sorcerer"": 1, ""Sniper"": 2, ""Vanguard"": 1}","{""Chrono"": 4, ""DarkStar"": 3, ""Set3_Brawler"": 2, ""Sniper"": 2}"


In [42]:
df_all_match_2

,gameid,gameduration,level,lastround,ranked,ingameduration,combination,champion,tier,player_cnt,season,user_id,active_synergy
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{""Cybernetic"": 1, ""Demolitionist"": 1, ""Infiltrator"": 1, ""Rebel"": 1, ""Set3_Brawler"": 1, ""Set3_Celestial"": 1, ""Set3_Void"": 1, ""Sniper"": 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum,8,season 3,KR-USER-1,{}
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{""Blaster"": 1, ""Chrono"": 1, ""Cybernetic"": 4, ""Demolitionist"": 1, ""Rebel"": 1, ""Set3_Blademaster"": 2, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 1, ""Set3_Void"": 1, ""Valkyrie"": 1, ""Vanguard"": 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum,8,season 3,KR-USER-2,"{""Cybernetic"": 3, ""Vanguard"": 2}"
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{""Blaster"": 1, ""Cybernetic"": 1, ""DarkStar"": 2, ""Infiltrator"": 1, ""Mercenary"": 1, ""Set3_Blademaster"": 1, ""Set3_Mystic"": 1, ""Valkyrie"": 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum,8,season 3,KR-USER-3,"{""Mercenary"": 1}"
3,KR_4291707834,1963.905273,7,38,2,1955.608521,"{""DarkStar"": 1, ""Protector"": 2, ""Set3_Blademaster"": 1, ""Set3_Celestial"": 5, ""Set3_Mystic"": 1, ""Sniper"": 1, ""StarGuardian"": 2, ""Vanguard"": 2}","{'Poppy': {'items': [], 'star': 2}, 'Xayah': {'items': [19, 23, 19], 'star': 3}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [16], 'star': 2}, 'Mordekaiser': {'items': [35, 67, 33], 'star': 3}, 'Ashe': {'items': [], 'star': 2}, 'Soraka': {'items': [68, 47], 'star': 2}}",platinum,8,season 3,KR-USER-4,"{""Protector"": 2, ""Set3_Celestial"": 4, ""Vanguard"": 2}"
4,KR_4291707834,1963.905273,8,38,1,1955.608521,"{""Blaster"": 1, ""Chrono"": 5, ""DarkStar"": 3, ""Protector"": 1, ""Set3_Blademaster"": 1, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 2, ""Sniper"": 2}","{'TwistedFate': {'items': [36, 27], 'star': 3}, 'Caitlyn': {'items': [49, 29], 'star': 2}, 'JarvanIV': {'items': [56], 'star': 2}, 'Blitzcrank': {'items': [15], 'star': 2}, 'Shen': {'items': [77, 6], 'star': 2}, 'Ezreal': {'items': [16], 'star': 2}, 'Lux': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-5,"{""Chrono"": 4, ""DarkStar"": 3, ""Set3_Sorcerer"": 2, ""Sniper"": 2}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
399993,KR_4357265434,2094.518555,7,33,4,1838.332764,"{""DarkStar"": 2, ""Demolitionist"": 2, ""Infiltrator"": 4, ""MechPilot"": 3, ""Set3_Sorcerer"": 2, ""Set3_Void"": 1, ""Valkyrie"": 1}","{'KhaZix': {'items': [67, 26, 56], 'star': 2}, 'KaiSa': {'items': [38, 44, 33], 'star': 2}, 'Annie': {'items': [], 'star': 3}, 'Shaco': {'items': [45, 29, 16], 'star': 3}, 'Rumble': {'items': [55, 66, 77], 'star': 2}, 'Lux': {'items': [46], 'star': 2}, 'Fizz': {'items': [], 'star': 2}}",challenger,8,season 3,KR-USER-399994,"{""Demolitionist"": 2, ""Infiltrator"": 4, ""MechPilot"": 3, ""Set3_Sorcerer"": 2}"
399994,KR_4357265434,2094.518555,8,33,5,1837.548096,"{""Blaster"": 1, ""Chrono"": 2, ""Cybernetic"": 6, ""Infiltrator"": 2, ""ManaReaver"": 2, ""Set3_Blademaster"": 3, ""Set3_Brawler"": 1, ""Vanguard"": 1}","{'Fiora': {'items': [35], 'star': 2}, 'Leona': {'items': [22], 'star': 1}, 'Shen': {'items': [], 'star': 1}, 'Lucian': {'items': [57, 2, 23], 'star': 2}, 'Vi': {'items': [67, 36], 'star': 2}, 'Irelia': {'items': [29, 28, 15], 'star': 2}, 'Thresh': {'items': [57], 'star': 2}, 'Ekko': {'items': [35, 15, 2], 'star': 2}}",challenger,8,season 3,KR-USER-399995,"{""Chrono"": 2, ""Cybernetic"": 6, ""I